<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/cis9557__baseline/blob/main/CIS9557_ConceptExamples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS9557 -- Concept Examples
## Business Analytics | Baruch College, Zicklin School of Business | Spring 2026

This notebook provides working Python implementations of fifteen exam review concepts from Modules 4, 5, and 6 of CIS9557. Each section contains an instructional description of the concept and its business context, followed by a code block that reproduces the example from the course reference document with printed results and visualizations.

**Exam scope:** Modules 4, 5, and 6 -- Provost & Fawcett, *Data Science for Business*, Ch. 3, 7, 8, 12, 13

---

---

| Module | Concept |
|--------|---------|
| **4** | 4.1 Association Rule Mining -- Support, Confidence, Lift |
| **4** | 4.2 Anomaly and Outlier Detection |
| **4** | 4.3 Clustering and Collaborative Filtering |
| **4** | 4.3 Supplement -- K-Nearest Neighbor (KNN) Classification |
| **4** | 4.4 Bias-Variance Tradeoff and Ensemble Methods |
| **4** | 4.5 Causal Inference vs. Correlation (Homophily) |
| **5** | 5.1 Expected Value Framework |
| **5** | 5.2 Confusion Matrix Components |
| **5** | 5.3 ROC Curve and AUC |
| **5** | 5.4 Cumulative Gains and Lift Charts |
| **5** | 5.5 Threshold Selection |
| **6** | 6.1 Six Data Quality Dimensions |
| **6** | 6.2 Data Wrangling Pipeline Stages |
| **6** | 6.3 Fitness for Purpose |
| **6** | 6.4 Chart Selection |
| **6** | 6.5 Alteryx Designer Cloud Pipeline Stages |

---


<b><center><font color=red><i>Run the next Setup cell first. All subsequent cells depend on the libraries and style
settings defined there.</i></font></center></b>

In [ ]:
# -- Setup: libraries and plot style ------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.spatial.distance import cosine as cosine_dist
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve,
    accuracy_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize

# Shared colour palette (course palette: navy / teal / mint)
NAVY  = '#0A1E35'
TEAL  = '#1C7293'
MINT  = '#4CC9A8'
AMBER = '#D97706'
RED   = '#DC2626'
GRAY  = '#6B7280'
LGRAY = '#E5E7EB'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8FAFC',
    'axes.edgecolor':   '#CBD5E1',
    'axes.grid':        True,
    'grid.color':       '#E2E8F0',
    'grid.linewidth':   0.6,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   12,
    'axes.labelsize':   10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
})

np.random.seed(42)
print("Setup complete.")

---
## Module 4 -- Ch. 12 & 13: Other Data Science Tasks

Concepts 4.1 through 4.5 cover association rule mining, anomaly detection,
clustering and collaborative filtering (including a KNN supplement),
ensemble methods, and the distinction between correlation and causal inference.

### Concept 4.1 -- Association Rule Mining: Support, Confidence, and Lift

**Business context:** Market basket analysis identifies product combinations that appear
together more frequently than chance would predict. Retailers use the resulting rules to
guide product placement, cross-sell promotions, and recommendation engines.

**Scenario:** A grocery chain analyses 1,000 transactions.
- 200 transactions contain both bread and butter.
- 300 transactions contain bread (with or without butter).
- 400 transactions contain butter (with or without bread).

**Three core metrics:**

| Metric | Formula | What it measures |
|--------|---------|-----------------|
| Support | count(A and B) / total | How often the pair appears overall |
| Confidence | count(A and B) / count(A) | How often B appears given A was purchased |
| Lift | confidence / support(B) | Whether A and B co-occur more than chance |

**Exam rule:** Lift > 1 signals a positive association. Lift = 1 signals independence.
Lift < 1 signals that the items are purchased together *less* often than chance predicts.

In [ ]:
# -- 4.1 Association Rule Mining -----------------------------------------------

# Transaction counts from the grocery scenario
total                  = 1_000
count_bread_and_butter =   200   # both items in the basket
count_bread            =   300   # bread present (with or without butter)
count_butter           =   400   # butter present (with or without bread)

# Three metrics
support    = count_bread_and_butter / total
confidence = count_bread_and_butter / count_bread
lift       = confidence / (count_butter / total)

# -- Printed results -----------------------------------------------------------
divider = '=' * 54
print(divider)
print('  ASSOCIATION RULE: bread --> butter')
print(divider)
print(f'  Support(bread AND butter)  = {count_bread_and_butter}/{total}  = {support:.4f}')
print(f'  Confidence(bread->butter)  = {count_bread_and_butter}/{count_bread} = {confidence:.4f}')
print(f'  Support(butter)            = {count_butter}/{total}  = {count_butter/total:.4f}')
print(f'  Lift                       = {confidence:.4f} / {count_butter/total:.4f} = {lift:.4f}')
print()
print(f'  A customer who buys bread is {(lift - 1)*100:.0f}% more likely')
print(f'  to also buy butter than a randomly chosen customer.')
print()

# Interpretation function (reusable in discussion)
def interpret_lift(lv):
    if lv > 1.0:
        return f'Positive association -- items co-occur more than chance (lift = {lv:.2f})'
    elif lv == 1.0:
        return f'Independent -- no relationship between items (lift = {lv:.2f})'
    else:
        return f'Avoidance -- items purchased together less than chance (lift = {lv:.2f})'

print('  Lift interpretation:', interpret_lift(lift))

# -- Bar chart of the three metrics -------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
labels  = ['Support', 'Confidence', 'Lift']
values  = [support, confidence, lift]
colors  = [TEAL, TEAL, MINT]

bars = ax.bar(labels, values, color=colors, width=0.45,
              edgecolor='white', linewidth=1.2)
ax.axhline(1.0, color=AMBER, linewidth=1.8, linestyle='--',
           label='Lift = 1.0  (independence baseline)')
ax.set_ylim(0, 2.2)
ax.set_ylabel('Metric value')
ax.set_title('Association Rule Mining: bread --> butter', fontweight='bold', color=NAVY)
ax.legend()

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', fontweight='bold', fontsize=11, color=NAVY)

plt.tight_layout()
plt.show()

### Concept 4.2 -- Anomaly and Outlier Detection

**Business context:** Anomaly detection separates observations that deviate materially
from established behavioural patterns. The challenge is distinguishing true anomalies
(warranting investigation) from noise (random variation) and rare-but-legitimate events
(consistent with known history).

**Scenario:** A bank processes 50,000 credit card transactions per day. Ninety-nine
percent of amounts fall between \$5 and \$500. One transaction appears for \$47,000 at
3:00 AM at an electronics retailer in a country the cardholder has never visited.

**Three classifications:**
- **Noise:** A \$3 rounding error on a \$120 transaction -- small, random, and without business consequence.
- **Rare but legitimate:** A \$4,200 jewelry purchase from a customer who buys jewelry once a year.
- **True anomaly:** The \$47,000 charge -- multiple simultaneous deviations with no historical precedent.

**Exam rule:** Classify the observation and cite the *specific dimensions of deviation*
(amount, time of day, geography). A z-score above 3 is the conventional threshold for
flagging an outlier in a normally distributed feature.

In [ ]:
# -- 4.2 Anomaly and Outlier Detection -----------------------------------------

# Generate 999 normal transactions (amount in dollars, clipped to $5-$500)
n_normal = 999
normal_amounts = np.random.normal(loc=120, scale=90, size=n_normal)
normal_amounts = np.clip(normal_amounts, 5, 500)

# The anomalous transaction
anomaly_amount = 47_000.0

all_amounts = np.append(normal_amounts, anomaly_amount)

# Z-score computed against the normal population
mu  = normal_amounts.mean()
sig = normal_amounts.std()

z_normal  = (normal_amounts - mu) / sig
z_anomaly = (anomaly_amount - mu) / sig

print('Normal transaction population:')
print(f'  Mean   = ${mu:,.2f}')
print(f'  Std    = ${sig:,.2f}')
print(f'  Min    = ${normal_amounts.min():,.2f}')
print(f'  Max    = ${normal_amounts.max():,.2f}')
print()
print('Flagged transaction:')
print(f'  Amount  = ${anomaly_amount:,.2f}')
print(f'  Z-score = {z_anomaly:.1f}  (threshold for flagging: +/- 3.0)')
print()
print('Classification: TRUE ANOMALY -- multiple simultaneous deviations')
print('  Dimension 1 (amount):       $47,000 vs typical range $5-$500')
print('  Dimension 2 (time of day):  3:00 AM')
print('  Dimension 3 (geography):    country never visited')

# -- Visualisation: histogram + anomaly marker ---------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: distribution of normal transactions
axes[0].hist(normal_amounts, bins=40, color=TEAL, edgecolor='white', alpha=0.85)
axes[0].axvline(mu, color=NAVY, linewidth=2, linestyle='--', label=f'Mean = ${mu:.0f}')
axes[0].axvline(mu + 3*sig, color=AMBER, linewidth=2,
                linestyle=':', label=f'+3 SD = ${mu+3*sig:.0f}')
axes[0].set_title('Normal Transaction Distribution', fontweight='bold', color=NAVY)
axes[0].set_xlabel('Transaction amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Panel 2: z-scores with anomaly highlighted
z_sample = z_normal[np.random.choice(n_normal, 200, replace=False)]
axes[1].scatter(range(200), sorted(z_sample), color=TEAL, s=14, alpha=0.6,
                label='Normal transactions (sample)')
axes[1].axhline(3,  color=AMBER, linewidth=1.5, linestyle='--', label='z = +3.0 threshold')
axes[1].axhline(-3, color=AMBER, linewidth=1.5, linestyle='--')
axes[1].scatter([200], [min(z_anomaly, 60)], color=RED, s=120, zorder=5,
                marker='*', label=f'Anomaly (z = {z_anomaly:.0f})')
axes[1].set_title('Z-Score Profile', fontweight='bold', color=NAVY)
axes[1].set_xlabel('Observation index (sorted)')
axes[1].set_ylabel('Z-score')
axes[1].set_ylim(-5, 65)
axes[1].legend()

plt.tight_layout()
plt.show()

### Concept 4.3 -- Clustering and Collaborative Filtering

**Business context:** Both techniques discover structure in data without requiring a
labelled target variable.

**Clustering -- streaming service segmentation:**
A streaming service segments 2 million subscribers using viewing behaviour
(hours watched per week, session frequency). No target variable such as churn is
specified. The resulting groups are exploratory behavioural profiles, not predictions.
The marketing team uses them to design segment-specific campaigns.

**Collaborative filtering -- film recommendation:**
A user has rated five action films highly and two romance films poorly. The system
identifies other users with a nearly identical rating pattern and recommends a thriller
those users rated highly -- even though the recommender knows nothing about the
thriller's attributes. Recommendations are driven entirely by behavioural similarity
in the user-item matrix.

**SVD (Singular Value Decomposition):** Compresses the large, sparse user-item
ratings matrix into a small number of latent factors that explain most of the
variation. This makes computation tractable on a dataset where most users have
not rated most items.

**Exam rules:**
- Clustering is exploratory. It produces no predicted label.
- Collaborative filtering relies on user-item interaction patterns, not item attributes.

In [ ]:
# -- 4.3a Clustering: streaming service viewer segmentation -------------------

# Synthetic viewer data: weekly hours watched and sessions per week
n_viewers = 300

# Three natural behavioural groups
casual    = np.column_stack([np.random.normal(4, 1.2, 100),
                              np.random.normal(3, 0.9, 100)])   # low hours, few sessions
moderate  = np.column_stack([np.random.normal(12, 2.0, 100),
                              np.random.normal(7, 1.5, 100)])   # mid hours
binge     = np.column_stack([np.random.normal(28, 3.5, 100),
                              np.random.normal(18, 3.0, 100)])  # high hours, many sessions

viewers = np.vstack([casual, moderate, binge])

# K-Means (k=3, exploratory -- NO target variable provided to the algorithm)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(viewers)

profile_names = {0: 'Casual', 1: 'Moderate', 2: 'Binge'}
# Map KMeans labels to meaningful names by cluster centroid ordering
order = np.argsort(kmeans.cluster_centers_[:, 0])
label_map = {order[i]: list(profile_names.values())[i] for i in range(3)}

fig, ax = plt.subplots(figsize=(8, 5))
palette = [TEAL, MINT, NAVY]
for k, col in zip(range(3), palette):
    mask = labels == k
    ax.scatter(viewers[mask, 0], viewers[mask, 1],
               color=col, s=28, alpha=0.65, label=label_map[k])

centroids = kmeans.cluster_centers_
ax.scatter(centroids[:, 0], centroids[:, 1],
           color=AMBER, s=180, zorder=5, marker='X', label='Cluster centroid')

ax.set_xlabel('Hours watched per week')
ax.set_ylabel('Sessions per week')
ax.set_title('K-Means Clustering: Viewer Segmentation (no target variable)',
             fontweight='bold', color=NAVY)
ax.legend(title='Segment')
plt.tight_layout()
plt.show()

print('Cluster centroids (hours/week, sessions/week):')
for i, (name, ctr) in enumerate(zip(profile_names.values(),
                                    centroids[order])):
    print(f'  {name:10s}: {ctr[0]:.1f} hours/week, {ctr[1]:.1f} sessions/week')
print()
print('Key point: the algorithm received ONLY hours and sessions.')
print('It produced segments, not predictions of any outcome.')

# -- 4.3b Collaborative Filtering: film recommendation -----------------------
print()
print('=' * 58)
print('  COLLABORATIVE FILTERING: film recommendation')
print('=' * 58)

# Small user-item ratings matrix (5 = loved, 0 = not seen)
films   = ['Action1', 'Action2', 'Action3', 'Action4', 'Action5',
           'Romance1', 'Romance2', 'Thriller1']
ratings = np.array([
    # A1  A2  A3  A4  A5  R1  R2  T1
    [  5,  4,  5,  4,  5,  1,  2,  0],   # Target user (has not seen Thriller1)
    [  5,  5,  4,  5,  4,  1,  1,  5],   # Similar user 1
    [  4,  4,  5,  4,  5,  2,  1,  4],   # Similar user 2
    [  1,  2,  1,  1,  2,  5,  5,  3],   # Dissimilar user (prefers romance)
    [  5,  4,  4,  5,  4,  1,  2,  5],   # Similar user 3
], dtype=float)

n_users = ratings.shape[0]
target  = ratings[0]

# Cosine similarity between target user and all others
sims = []
for i in range(1, n_users):
    # Compare only films both users have rated (non-zero)
    mask = (target > 0) & (ratings[i] > 0)
    if mask.sum() > 0:
        sim = 1 - cosine_dist(target[mask], ratings[i][mask])
    else:
        sim = 0.0
    sims.append((i, sim))

print()
sim_df = pd.DataFrame(sims, columns=['User index', 'Cosine similarity'])
sim_df['Label'] = ['Similar user 1', 'Similar user 2',
                   'Dissimilar user', 'Similar user 3']
print(sim_df[['Label', 'Cosine similarity']].to_string(index=False))

# Weighted average score for Thriller1 (index 7) from similar users
thriller_idx  = 7
weighted_sum  = sum(sim * ratings[i][thriller_idx]
                    for i, sim in sims if ratings[i][thriller_idx] > 0)
weight_total  = sum(sim for i, sim in sims if ratings[i][thriller_idx] > 0)
predicted_score = weighted_sum / weight_total

print()
print(f'  Target user has NOT seen Thriller1.')
print(f'  Predicted rating for Thriller1: {predicted_score:.2f} / 5.00')
print(f'  --> Recommend Thriller1 based on behavioural similarity.')
print()
print('  Note: the recommendation uses user-rating patterns only.')
print('  No information about Thriller1 as a film was consulted.')

### Concept 4.3 Supplement -- K-Nearest Neighbor (KNN) Classification

**Relationship to Concept 4.3 -- Clustering:** K-Nearest Neighbor classification
uses the same distance-based similarity logic as clustering but applies it to a
*supervised* task. The algorithm estimates no parameters: it stores the entire
training set and, at prediction time, finds the K most similar labeled training
observations and assigns the majority class among them.

**Contrast with unsupervised clustering:**

| Characteristic | Clustering (Concept 4.3) | KNN Classification (this supplement) |
|----------------|--------------------------|--------------------------------------|
| Target variable | None -- exploratory | Required -- labeled training cases |
| Output | Segment membership | Class prediction for a new observation |
| Basis | Density or distance | Distance to stored labeled training cases |

**Business context:** A bank predicts whether a new loan applicant will default.
For any new applicant, KNN locates the K most similar past applicants (by age and
income, after feature scaling) and predicts the outcome the majority of those
neighbors experienced.

**Why feature scaling is required before any distance calculation:**
Annual income may range from $20,000 to $150,000 (spread: $130,000). Age may range
from 22 to 65 (spread: 43). Without scaling, a $10,000 income difference dwarfs any
age difference -- income dominates every distance calculation regardless of its actual
predictive value. StandardScaler normalises each feature to mean = 0 and
standard deviation = 1 before distances are computed.

**The K selection tradeoff -- bridge to Concept 4.4:**

| K value | Decision boundary | Train accuracy | Bias-Variance diagnosis |
|---------|-----------------|----------------|------------------------|
| K = 1 | Highly irregular | 100% (memorises training data) | High variance -- overfitting |
| K = 5 | Moderate | High | Balanced -- generalises well |
| K = 25 | Smooth | Lower | Higher bias -- boundary oversmoothed |

At K = 1, the model is its own nearest neighbor on the training set: training
accuracy is exactly 1.000. Small changes in the training data produce large changes
in predictions. This is the high-variance regime that Concept 4.4 addresses directly.

**Exam rules:**
- KNN is supervised: it requires labeled training observations.
- Feature scaling is required before any distance-based computation.
- Small K produces high variance (irregular boundary, sensitive to noise).
- Large K produces higher bias (oversmoothed boundary, misses local structure).
- The training data itself constitutes the model -- no coefficients are estimated.

In [ ]:
# -- 4.3 Supplement: K-Nearest Neighbor (KNN) Classification ------------------
# KNN-specific imports kept here to make this cell self-contained.
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# -- Dataset: synthetic loan applicant data ------------------------------------
# Two features only so that the distance mechanism is easy to trace.
# make_classification is imported in the Setup cell.
X_knn, y_knn = make_classification(
    n_samples=1000, n_features=2, n_informative=2,
    n_redundant=0, n_repeated=0, n_classes=2,
    weights=[0.80, 0.20], class_sep=1.2, random_state=17
)

# Rescale to interpretable business units before demonstrating the scaling step.
# Feature 0: Age proxy   (range approx 22 to 65)
# Feature 1: Income proxy (range approx $20,000 to $150,000)
X_knn[:, 0] = X_knn[:, 0] * 8 + 43
X_knn[:, 1] = X_knn[:, 1] * 25000 + 80000

X_tr_k, X_ts_k, y_tr_k, y_ts_k = train_test_split(
    X_knn, y_knn, test_size=0.25, random_state=42, stratify=y_knn
)

# -- Feature scaling demonstration ---------------------------------------------
print('Feature scaling demonstration:')
print()
print('BEFORE scaling (raw features):')
print(f'  Age proxy    -- range: {X_tr_k[:,0].min():.0f} to {X_tr_k[:,0].max():.0f}  '
      f'(spread: {X_tr_k[:,0].max()-X_tr_k[:,0].min():.0f} units)')
print(f'  Income proxy -- range: ${X_tr_k[:,1].min():,.0f} to ${X_tr_k[:,1].max():,.0f}  '
      f'(spread: ${X_tr_k[:,1].max()-X_tr_k[:,1].min():,.0f})')
print()
print('Without scaling, income dominates every distance calculation.')
print('A $10,000 income difference dwarfs any age difference -- age is ignored.')
print()

scaler_k  = StandardScaler()
X_tr_k_sc = scaler_k.fit_transform(X_tr_k)
X_ts_k_sc = scaler_k.transform(X_ts_k)   # transform only -- never re-fit on test data

print('AFTER StandardScaler (mean=0, std=1):')
print(f'  Age proxy    -- range: {X_tr_k_sc[:,0].min():.2f} to {X_tr_k_sc[:,0].max():.2f}')
print(f'  Income proxy -- range: {X_tr_k_sc[:,1].min():.2f} to {X_tr_k_sc[:,1].max():.2f}')
print()
print('Both features now measured in comparable standardised units.')
print('A one-unit scaled difference in Age equals one-unit scaled difference in Income.')

# -- Fit K=5 and trace a single prediction ------------------------------------
print()
print('=' * 58)
print('K=5 Classifier: single-prediction trace')
print('=' * 58)

knn5 = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
knn5.fit(X_tr_k_sc, y_tr_k)

# New applicant: Age 43, Income $88,000
new_applicant = np.array([[43.0, 88000.0]])
new_scaled    = scaler_k.transform(new_applicant)

prediction     = knn5.predict(new_scaled)[0]
distances, idx = knn5.kneighbors(new_scaled)

print()
print('New applicant: Age = 43, Income = $88,000')
print()
print('5 nearest training neighbors:')
print(f'  {"Rank":>4}  {"Scaled dist":>12}  {"Age":>6}  {"Income":>12}  {"Default":>8}')
print('  ' + '-' * 52)
for rank, (dist, i) in enumerate(zip(distances[0], idx[0]), 1):
    age_val = X_tr_k[i, 0]
    inc_val = X_tr_k[i, 1]
    outcome = 'YES' if y_tr_k[i] == 1 else 'no'
    print(f'  {rank:>4}  {dist:>12.4f}  {age_val:>6.0f}  ${inc_val:>11,.0f}  {outcome:>8}')

n_default = int(sum(y_tr_k[i] for i in idx[0]))
majority  = 'DEFAULT' if n_default >= 3 else 'NO DEFAULT'
print()
print(f'  Neighbor vote: {n_default} default, {5 - n_default} no default')
print(f'  Predicted outcome for this applicant: {majority}')
print()
print('  This trace shows the complete reasoning path.')
print('  There is no formula or coefficient -- the prediction follows')
print('  entirely from stored training data and the distance metric.')

# -- Confusion matrix and derived metrics at K=5 ------------------------------
y_pred_k5 = knn5.predict(X_ts_k_sc)
cm_k      = confusion_matrix(y_ts_k, y_pred_k5)
acc_k5    = accuracy_score(y_ts_k, y_pred_k5)
prec_k5   = precision_score(y_ts_k, y_pred_k5, zero_division=0)
rec_k5    = recall_score(y_ts_k, y_pred_k5, zero_division=0)

print()
print('=' * 58)
print('Confusion matrix on hold-out test set (K=5)')
print('=' * 58)
print()
print(f'                    Predicted No Default   Predicted Default')
print(f'  Actual No Default     TN = {cm_k[0,0]:>3}               FP = {cm_k[0,1]:>3}')
print(f'  Actual Default        FN = {cm_k[1,0]:>3}               TP = {cm_k[1,1]:>3}')
print()
print(f'  Accuracy:   {acc_k5:.3f}  ({acc_k5*100:.1f}%)')
print(f'  Precision:  {prec_k5:.3f}  ({prec_k5*100:.1f}% of predicted defaults are actual defaults)')
print(f'  Recall:     {rec_k5:.3f}  ({rec_k5*100:.1f}% of actual defaults are caught)')
print()
print('  Precision and recall reveal minority-class performance.')
print('  Accuracy alone, as shown in Concept 5.2, can mislead on imbalanced data.')

# -- Effect of K: train vs. test accuracy at K=1, K=5, K=25 ------------------
print()
print('=' * 58)
print('Effect of K on train vs. test accuracy')
print('=' * 58)
print()

k_values   = [1, 5, 25]
train_accs = []
test_accs  = []

for k in k_values:
    mk = KNeighborsClassifier(n_neighbors=k, metric='minkowski', p=2)
    mk.fit(X_tr_k_sc, y_tr_k)
    train_accs.append(accuracy_score(y_tr_k, mk.predict(X_tr_k_sc)))
    test_accs.append(accuracy_score(y_ts_k,  mk.predict(X_ts_k_sc)))

for k, tr, ts in zip(k_values, train_accs, test_accs):
    gap  = tr - ts
    if gap > 0.08:
        diag = 'High variance (overfitting)'
    elif ts < 0.75:
        diag = 'High bias'
    else:
        diag = 'Balanced'
    print(f'  K={k:>2}: Train={tr:.3f}  Test={ts:.3f}  Gap={gap:.3f}  --> {diag}')

print()
print('K=1: training accuracy = 1.000 (the model memorises every training case).')
print('This is the highest-variance setting. As K increases, the boundary smooths,')
print('variance decreases, and bias begins to rise. See Concept 4.4 for the full')
print('treatment of the bias-variance tradeoff.')

# -- Chart: K effect on train vs. test accuracy --------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
x_pos = np.arange(len(k_values))
width = 0.32

b1 = ax.bar(x_pos - width/2, train_accs, width, color=TEAL, alpha=0.88,
            edgecolor='white', label='Train accuracy')
b2 = ax.bar(x_pos + width/2, test_accs,  width, color=MINT, alpha=0.88,
            edgecolor='white', label='Test accuracy')

ax.set_xticks(x_pos)
ax.set_xticklabels([f'K = {k}' for k in k_values], fontsize=11)
ax.set_ylim(0.55, 1.10)
ax.set_ylabel('Accuracy')
ax.set_title('KNN: Effect of K on Train vs. Test Accuracy\n'
             'Small K = high variance   |   Large K = higher bias',
             fontweight='bold', color=NAVY)
ax.legend()

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9,
            fontweight='bold', color=NAVY)

# Annotate K=1 overfitting
ax.annotate('Train = 1.000\n(memorises training data)',
            xy=(x_pos[0] - width / 2, train_accs[0]),
            xytext=(x_pos[0] + 0.05, train_accs[0] - 0.09),
            fontsize=8, color=RED,
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.2))

plt.tight_layout()
plt.show()

### Concept 4.4 -- Bias-Variance Tradeoff and Ensemble Methods

**Business context:** All predictive models face a fundamental tradeoff between
bias (systematic error from oversimplification) and variance (sensitivity to
fluctuations in the training data). Ensemble methods manage this tradeoff.

**Scenario:** A bank trains a loan default prediction model on historical applicant data.

**Four model behaviours:**
- **High bias:** A model that predicts "no default" for every applicant learns almost
  nothing. Its error is systematic and consistent.
- **High variance:** A model that memorises the training set achieves near-zero training
  error but performs erratically on new applicants (overfitting).
- **Bagging (Random Forest):** Trains many trees on bootstrap samples and averages their
  predictions. Averaging reduces variance without increasing bias.
- **Boosting (Gradient Boosting):** Trains trees sequentially, each correcting the
  residual errors of the previous one. Reduces both bias and variance iteratively.

**Exam rules:**
- Bagging targets variance.
- Boosting targets residual error through sequential correction.
- Random forests add feature randomization on top of bagging.

In [ ]:
# -- 4.4 Bias-Variance Tradeoff and Ensemble Methods --------------------------

# Synthetic loan default dataset (binary classification)
X, y = make_classification(
    n_samples=2000, n_features=15, n_informative=8,
    n_redundant=4, n_classes=2, weights=[0.85, 0.15],
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Four model configurations
models = {
    'High Bias\n(Logistic Reg. C=0.001)': LogisticRegression(C=0.001, max_iter=1000),
    'High Variance\n(Deep Decision Tree)': DecisionTreeClassifier(max_depth=None),
    'Bagging\n(Random Forest)':            RandomForestClassifier(
                                               n_estimators=100, random_state=42),
    'Boosting\n(Gradient Boosting)':       GradientBoostingClassifier(
                                               n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc  = accuracy_score(y_test,  model.predict(X_test))
    results[name] = {'Train accuracy': train_acc, 'Test accuracy': test_acc}
    gap_label = 'HIGH VARIANCE' if (train_acc - test_acc) > 0.05 else (
                'HIGH BIAS'    if test_acc < 0.80 else 'WELL-CALIBRATED')
    print(f'{name.replace(chr(10), " "):42s}  '
          f'Train={train_acc:.3f}  Test={test_acc:.3f}  --> {gap_label}')

# -- Bar chart: train vs test accuracy ----------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))
short_names = ['High Bias', 'High Variance', 'Bagging\n(RF)', 'Boosting\n(GB)']
x = np.arange(len(short_names))
width = 0.36

train_vals = [results[k]['Train accuracy'] for k in results]
test_vals  = [results[k]['Test accuracy']  for k in results]

b1 = ax.bar(x - width/2, train_vals, width, label='Train accuracy',
            color=TEAL, alpha=0.85, edgecolor='white')
b2 = ax.bar(x + width/2, test_vals,  width, label='Test accuracy',
            color=MINT, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=9)
ax.set_ylim(0.60, 1.02)
ax.set_ylabel('Accuracy')
ax.set_title('Bias-Variance Tradeoff: Train vs. Test Accuracy by Model Type',
             fontweight='bold', color=NAVY)
ax.legend()
ax.axhline(0.85, color=AMBER, linewidth=1.2, linestyle=':', alpha=0.7,
           label='0.85 reference')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', fontsize=8, color=NAVY)

plt.tight_layout()
plt.show()

print()
print('Interpretation:')
print('  High Bias:       both train and test accuracy are low -- the model')
print('                   has not learned the data well.')
print('  High Variance:   train accuracy is near 1.0 but test accuracy drops')
print('                   -- the model memorised the training set (overfitting).')
print('  Random Forest:   train-test gap is small -- averaging reduces variance.')
print('  Gradient Boost:  sequential correction reduces both bias and variance.')

### Concept 4.5 -- Causal Inference vs. Correlation (Homophily)

**Business context:** Correlated behaviour in a social network does not imply that
one person's behaviour *caused* another's. Homophily -- the tendency for similar
people to connect -- produces correlation without causation. Acting on a correlated
but non-causal feature wastes budget on people who would have converted regardless.

**Scenario:** A social media platform observes that users connected to early adopters
of a new feature are significantly more likely to adopt it. The platform concludes
that social influence is driving adoption and launches a targeted campaign.

**The flaw:** Connected users may share the same demographic profile and interests
(homophily). They adopt at nearly the same time not because one influenced the other,
but because they were already alike.

**Corrective test:** A randomised experiment assigns some friend-pairs to receive the
campaign and others to a control group. If treated pairs adopt at a significantly
higher rate, influence is real. Observational correlation alone cannot establish this.

**Exam rule:** Identify whether the scenario describes homophily or genuine influence,
and state what additional evidence (a randomised experiment) would be required.

In [ ]:
# -- 4.5 Homophily vs. Causal Influence ----------------------------------------

# Simulation setup
n_users       = 1_000
early_pct     = 0.15      # 15% are "early adopters" by disposition

# Each user has a latent "adoption propensity" score in [0, 1]
propensity = np.random.beta(a=2, b=5, size=n_users)  # right-skewed: most users unlikely

# Designate early adopters as those with the highest propensity
threshold   = np.percentile(propensity, 100 * (1 - early_pct))
is_early    = propensity >= threshold

# Build a homophilous network: users are more likely to be "friends"
# with others who share similar propensity (similar people connect)
# For simplicity: each user's friends are drawn from nearby propensity ranks
ranks = np.argsort(propensity)
rank_of = np.argsort(ranks)  # rank_of[i] = propensity rank of user i

def has_early_friend(user_idx, window=60):
    # Returns True if any user within +/- window propensity ranks is an early adopter.
    lo = max(0, rank_of[user_idx] - window)
    hi = min(n_users, rank_of[user_idx] + window)
    neighbours = ranks[lo:hi]
    return any(is_early[neighbours])

# Identify non-adopters who have at least one early-adopter connection
non_early = ~is_early
connected_to_early = np.array([has_early_friend(i) for i in range(n_users)])

# Measure: average propensity of (a) non-adopters connected to early adopters
#           vs (b) non-adopters NOT connected to early adopters
grp_connected    = propensity[non_early &  connected_to_early]
grp_unconnected  = propensity[non_early & ~connected_to_early]

print('Propensity scores (proxy for likelihood of adoption):')
print(f'  Non-adopters with an early-adopter connection:    '
      f'mean = {grp_connected.mean():.3f}  (n={len(grp_connected)})')
print(f'  Non-adopters without an early-adopter connection: '
      f'mean = {grp_unconnected.mean():.3f}  (n={len(grp_unconnected)})')
print()
print('Observation: the connected group has a HIGHER average propensity.')
print('This is homophily -- similar people form connections.')
print('The correlation is real; the causal inference is not.')
print()

# Simulate a campaign: treated group = connected non-adopters
# Without a randomised control, we cannot isolate the campaign effect
naive_lift      = grp_connected.mean() / grp_unconnected.mean()
rct_effect_size = 0.04   # true incremental causal effect (small)

print(f'Naive (observational) apparent lift:  {naive_lift:.2f}x')
print(f'RCT-estimated true causal effect:     {rct_effect_size:.2f} percentage points')
print(f'Overstated causal estimate:           {naive_lift - 1:.2f}x  (homophily inflates it)')

# -- Distribution comparison chart --------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, 1, 30)

ax.hist(grp_unconnected, bins=bins, color=GRAY,  alpha=0.65,
        label='No early-adopter connection')
ax.hist(grp_connected,   bins=bins, color=TEAL,  alpha=0.75,
        label='Has early-adopter connection')
ax.axvline(grp_connected.mean(),   color=NAVY,  lw=2, linestyle='-')
ax.axvline(grp_unconnected.mean(), color=AMBER, lw=2, linestyle='--')
ax.set_xlabel('Adoption propensity score')
ax.set_ylabel('Number of users')
ax.set_title('Homophily: Connected Users Have Higher Propensity by Disposition\n'
             '(not because of influence)', fontweight='bold', color=NAVY)
ax.legend()
plt.tight_layout()
plt.show()

print()
print('Corrective test required: randomised experiment.')
print('Randomly assign friend-pairs to (a) receive campaign or (b) control.')
print('Only a significant difference in adoption rates between groups')
print('confirms genuine causal influence.')

---
## Module 5 -- Ch. 7 & 8: Decision Analytic Thinking and Visualizing Model Performance

Concepts 5.1 through 5.5 cover the expected value framework, confusion matrix
components, ROC curves and AUC, cumulative gains and lift charts, and threshold
selection. This module has the highest density of quantitative content on the exam.

### Concept 5.1 -- Expected Value Framework

**Business context:** A model's value to the organisation depends not only on its
accuracy but on the *financial consequences* of each type of correct and incorrect
prediction. The expected value framework makes this explicit by attaching a dollar
value to each cell of the confusion matrix.

**Scenario:** A bank model flags fraudulent credit card transactions.

| Outcome | Consequence | Value |
|---------|-------------|-------|
| True Positive (fraud correctly flagged) | Loss prevented | +\$200 |
| False Positive (legitimate transaction blocked) | Customer friction | --\$10 |
| False Negative (fraud missed) | Full loss absorbed | --\$200 |
| True Negative (legitimate correctly passed) | No action needed | \$0 |

**Model results on 1,000 test transactions:** 40 TP, 10 FN, 95 FP, 855 TN.

**Exam rule:** Apply the correct cost or benefit to each confusion matrix cell.
Sum across all cells. A positive expected value confirms the model adds value
over not scoring at all.

In [ ]:
# -- 5.1 Expected Value Framework ----------------------------------------------

# Confusion matrix cell counts
TP, FN, FP, TN = 40, 10, 95, 855
total = TP + FN + FP + TN

# Cost-benefit values (dollars)
benefit_TP =  200   # fraud prevented
cost_FP    =  -10   # customer friction
cost_FN    = -200   # full fraud loss
value_TN   =    0   # no action, no cost

# Expected value calculation
ev_TP = TP * benefit_TP
ev_FP = FP * cost_FP
ev_FN = FN * cost_FN
ev_TN = TN * value_TN
ev_total = ev_TP + ev_FP + ev_FN + ev_TN

print('Confusion matrix:')
print(f'                  Predicted Fraud   Predicted Legitimate')
print(f'  Actual Fraud     TP = {TP:3d}             FN = {FN:3d}')
print(f'  Actual Legit.    FP = {FP:3d}             TN = {TN:3d}')
print()
print('Expected value calculation:')
print(f'  TP contribution: {TP:3d} x ${benefit_TP:>6,} = ${ev_TP:>8,}')
print(f'  FN contribution: {FN:3d} x ${cost_FN:>6,} = ${ev_FN:>8,}')
print(f'  FP contribution: {FP:3d} x ${cost_FP:>6,} = ${ev_FP:>8,}')
print(f'  TN contribution: {TN:3d} x ${value_TN:>6,} = ${ev_TN:>8,}')
print(f'  {"-"*42}')
print(f'  Total EV across {total:,} transactions: ${ev_total:>8,}')
print(f'  EV per transaction scored:           ${ev_total/total:>8.2f}')
print()
baseline_ev = 0
print(f'  Baseline EV (no model):              ${baseline_ev:>8,}')
print(f'  Model adds value: {ev_total > baseline_ev}')

# -- Waterfall chart ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
components = ['TP\n(fraud\nprevented)', 'FN\n(fraud\nmissed)',
              'FP\n(blocked\nlegit.)', 'TN\n(no action)', 'Total EV']
amounts    = [ev_TP, ev_FN, ev_FP, ev_TN, ev_total]
colors     = [MINT, RED, AMBER, GRAY, TEAL]

bars = ax.bar(components, amounts, color=colors, edgecolor='white', linewidth=1.1)
ax.axhline(0, color=NAVY, linewidth=1.2, linestyle='-')
ax.set_ylabel('Dollar value ($)')
ax.set_title('Expected Value by Confusion Matrix Component',
             fontweight='bold', color=NAVY)

for bar, val in zip(bars, amounts):
    va = 'bottom' if val >= 0 else 'top'
    offset = 60 if val >= 0 else -60
    ax.text(bar.get_x() + bar.get_width()/2, val + offset,
            f'${val:,}', ha='center', fontsize=9, fontweight='bold', color=NAVY)

plt.tight_layout()
plt.show()

### Concept 5.2 -- Confusion Matrix Components

**Business context:** Accuracy alone is an unreliable metric whenever positive cases
are rare. A more complete picture requires precision, recall, and specificity -- each
measuring a different facet of model performance.

**Scenario:** A hospital screens 100 patients for a rare infection. 10 actually have it.
Model results: 8 TP, 2 FN, 18 FP, 72 TN.

| Metric | Formula | Result | Plain meaning |
|--------|---------|--------|---------------|
| Accuracy | (TP+TN)/total | 80% | Correct on 80 of 100 cases |
| Precision | TP/(TP+FP) | 30.8% | Of flagged patients, 31% truly infected |
| Recall | TP/(TP+FN) | 80% | Model catches 80% of actual infections |
| Specificity | TN/(TN+FP) | 80% | Correctly clears 80% of healthy patients |

**Why accuracy misleads:** Predicting "negative" for all 100 patients yields 90%
accuracy -- yet catches zero infections. In imbalanced datasets, accuracy inflates
by reflecting the dominant class, not model utility.

**Exam rule:** Calculate all four metrics from cell counts. Explain why accuracy
is insufficient when positives are rare.

In [ ]:
# -- 5.2 Confusion Matrix Components ------------------------------------------

# Cell counts from the hospital infection scenario
TP, FN, FP, TN = 8, 2, 18, 72
total = TP + FN + FP + TN

# Four metrics
accuracy    = (TP + TN) / total
precision   = TP / (TP + FP)
recall      = TP / (TP + FN)       # also called sensitivity
specificity = TN / (TN + FP)

print('Confusion matrix (hospital infection screening, 100 patients):')
print(f'                    Predicted Positive  Predicted Negative')
print(f'  Actual Positive      TP = {TP:2d}                FN = {FN:2d}')
print(f'  Actual Negative      FP = {FP:2d}                TN = {TN:2d}')
print()
print(f'  Accuracy:    (TP+TN)/total = ({TP}+{TN})/{total}  = {accuracy:.3f}  ({accuracy*100:.1f}%)')
print(f'  Precision:   TP/(TP+FP)   = {TP}/({TP}+{FP}) = {precision:.3f}  ({precision*100:.1f}%)')
print(f'  Recall:      TP/(TP+FN)   = {TP}/({TP}+{FN})  = {recall:.3f}  ({recall*100:.1f}%)')
print(f'  Specificity: TN/(TN+FP)   = {TN}/({TN}+{FP}) = {specificity:.3f}  ({specificity*100:.1f}%)')

# -- The accuracy trap: predict all-negative --------------------------------
acc_all_neg = TN / total
print()
print('The accuracy trap:')
print(f'  If the model predicts NEGATIVE for all {total} patients:')
print(f'  Accuracy = {TN}/{total} = {acc_all_neg:.1%}  (higher than the real model!)')
print(f'  Infections caught = 0 / {TP+FN}  (clinically useless)')

# -- Confusion matrix heatmap + metric bar chart ------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Panel 1: heatmap
cm_array = np.array([[TP, FN], [FP, TN]])
im = axes[0].imshow(cm_array, cmap='YlGnBu', aspect='auto')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Predicted\nPositive', 'Predicted\nNegative'])
axes[0].set_yticklabels(['Actual\nPositive', 'Actual\nNegative'])
axes[0].set_title('Confusion Matrix', fontweight='bold', color=NAVY)
labels_cm = [[f'TP = {TP}', f'FN = {FN}'], [f'FP = {FP}', f'TN = {TN}']]
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, labels_cm[i][j], ha='center', va='center',
                     fontsize=13, fontweight='bold', color=NAVY)

# Panel 2: metric bars
metric_names = ['Accuracy', 'Precision', 'Recall', 'Specificity']
metric_vals  = [accuracy, precision, recall, specificity]
bar_colors   = [AMBER, TEAL, MINT, TEAL]

bars = axes[1].bar(metric_names, metric_vals, color=bar_colors,
                   edgecolor='white', linewidth=1.1)
axes[1].set_ylim(0, 1.12)
axes[1].set_ylabel('Metric value')
axes[1].set_title('Derived Metrics', fontweight='bold', color=NAVY)
for bar, val in zip(bars, metric_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.1%}', ha='center', fontsize=10, fontweight='bold', color=NAVY)

plt.tight_layout()
plt.show()

### Concept 5.3 -- ROC Curve and AUC

**Business context:** The ROC curve evaluates a classifier's discriminative ability
across *all possible* classification thresholds simultaneously, making it independent
of any particular operating threshold. AUC (Area Under the Curve) summarises this
as a single number.

**Scenario:** Two models predict customer churn.
- **Model A:** AUC = 0.85 -- strong overall discriminator.
- **Model B:** AUC = 0.72 -- weaker overall but tighter control of false positives.

**Reading the ROC curve:**
- X-axis: False Positive Rate (FPR = FP / (FP + TN))
- Y-axis: True Positive Rate (TPR = TP / (TP + FN))
- Diagonal (TPR = FPR): random model, AUC = 0.50
- Upper-left bow: strong model

**AUC interpretation:** AUC of 0.85 means the model ranks a randomly selected
churner higher than a randomly selected non-churner 85% of the time.

**Exam rule:** AUC is threshold-independent. A model with AUC = 0.50 is no
better than random guessing.

In [ ]:
# -- 5.3 ROC Curve and AUC ----------------------------------------------------

# Generate synthetic churn scores for two models
n = 2_000
y_true = np.where(np.random.rand(n) < 0.20, 1, 0)   # 20% churn rate

# Model A: strong discriminator (AUC ~ 0.85)
scores_a = np.where(y_true == 1,
                    np.random.beta(5, 2, n),
                    np.random.beta(2, 5, n))

# Model B: weaker overall, tighter on FPR (AUC ~ 0.72)
scores_b = np.where(y_true == 1,
                    np.random.beta(3, 3, n),
                    np.random.beta(2, 4, n))

# ROC curves
fpr_a, tpr_a, _ = roc_curve(y_true, scores_a)
fpr_b, tpr_b, _ = roc_curve(y_true, scores_b)
auc_a = auc(fpr_a, tpr_a)
auc_b = auc(fpr_b, tpr_b)

print(f'Model A AUC: {auc_a:.3f}  (target: ~0.85)')
print(f'Model B AUC: {auc_b:.3f}  (target: ~0.72)')
print()
print('AUC interpretation:')
print(f'  Model A ranks a random churner above a random non-churner')
print(f'  {auc_a*100:.1f}% of the time.')
print()
print('Choosing between the two models:')
print('  If false alarms are costly (calling non-churners wastes budget),')
print('  Model B may be preferred at a low FPR operating point.')
print('  If capturing every churner is paramount, Model A is preferred.')

# -- ROC plot -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(fpr_a, tpr_a, color=TEAL, linewidth=2.5,
        label=f'Model A (AUC = {auc_a:.2f}) -- stronger overall')
ax.plot(fpr_b, tpr_b, color=MINT,  linewidth=2.5, linestyle='--',
        label=f'Model B (AUC = {auc_b:.2f}) -- lower FPR at chosen point')
ax.plot([0, 1], [0, 1], color=GRAY, linewidth=1.5, linestyle=':',
        label='Random model (AUC = 0.50)')

ax.fill_between(fpr_a, tpr_a, alpha=0.08, color=TEAL)
ax.fill_between(fpr_b, tpr_b, alpha=0.06, color=MINT)

ax.set_xlabel('False Positive Rate  (FPR = FP / (FP + TN))')
ax.set_ylabel('True Positive Rate   (TPR = TP / (TP + FN))')
ax.set_title('ROC Curves: Churn Prediction Models A and B',
             fontweight='bold', color=NAVY)
ax.legend(loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

ax.annotate('Upper-left: strong model', xy=(0.10, 0.88),
            fontsize=9, color=NAVY, fontstyle='italic')
ax.annotate('Diagonal: random model\n(AUC = 0.50)', xy=(0.55, 0.42),
            fontsize=9, color=GRAY, fontstyle='italic')

plt.tight_layout()
plt.show()

### Concept 5.4 -- Cumulative Gains and Lift Charts

**Business context:** Gains and lift charts translate model performance into a
direct business decision: how many customers to contact to maximise return on a
direct marketing campaign.

**Scenario:** A catalog retailer has 10,000 customers and a 10% historical response
rate (1,000 actual responders). The model scores and ranks all customers.

**Cumulative gains at depth 10%:** The top-scored 10% of the list (1,000 customers)
contains 300 of the 1,000 responders. Lift = 300 / 100 = **3.0** -- the model is
three times better than random selection at this depth.

**Break-even analysis:** Mailing cost = \$2 per piece, margin per response = \$30.
Break-even response rate = 2/30 = 6.7%. Top decile rate = 30%, well above break-even.

**Exam rule:** Lift at depth d = (fraction of positives captured) / d.
At depth 100% (entire list), lift must equal 1.0.

In [ ]:
# -- 5.4 Cumulative Gains and Lift Charts -------------------------------------

# Reproduce the catalog retailer scenario
n_customers   = 10_000
response_rate = 0.10
n_responders  = int(n_customers * response_rate)   # 1,000 actual responders

# Generate customer scores and true response labels
y_true_retail = np.zeros(n_customers, dtype=int)
y_true_retail[:n_responders] = 1
np.random.shuffle(y_true_retail)

# Model scores: responders get higher scores (realistic signal)
scores_retail = np.where(
    y_true_retail == 1,
    np.random.beta(6, 2, n_customers),   # responders: higher probability
    np.random.beta(2, 6, n_customers),   # non-responders: lower probability
)

# Sort by descending predicted probability (as a marketer would)
sort_idx = np.argsort(-scores_retail)
y_sorted = y_true_retail[sort_idx]

# Cumulative gains at each depth
depths      = np.linspace(0, 1, 201)
n_contacted = (depths * n_customers).astype(int)
gains       = np.array([y_sorted[:nc].sum() / n_responders for nc in n_contacted])
lift        = np.where(depths > 0, gains / depths, 1.0)

# -- Key figures at specified depths ------------------------------------------
report_depths = [0.10, 0.20, 0.40, 1.00]
print(f'{"Depth":>8}  {"Customers contacted":>20}  {"Responders captured":>22}  {"Lift":>6}')
print('-' * 64)
for d in report_depths:
    idx   = int(d * 200)
    nc    = n_contacted[idx]
    cap   = int(gains[idx] * n_responders)
    lv    = lift[idx]
    print(f'{d:>8.0%}  {nc:>20,}  {cap:>22,}  ({gains[idx]:.0%})  {lv:>6.2f}')

# Break-even analysis
cost_per_piece   = 2.00
margin_per_resp  = 30.00
breakeven_rate   = cost_per_piece / margin_per_resp
top_decile_rate  = gains[int(0.10 * 200)]
actual_rr_top10  = top_decile_rate * n_responders / (0.10 * n_customers)

print()
print(f'Break-even response rate: ${cost_per_piece:.2f} / ${margin_per_resp:.2f} = {breakeven_rate:.1%}')
print(f'Top-decile response rate with model:                {actual_rr_top10:.1%}')
print(f'Decision: mail the top decile -- response rate ({actual_rr_top10:.1%}) '
      f'exceeds break-even ({breakeven_rate:.1%}).')

# -- Two-panel chart: gains + lift -------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Gains chart
ax1.plot(depths * 100, gains * 100, color=TEAL, linewidth=2.5, label='Model')
ax1.plot([0, 100], [0, 100], color=GRAY, linewidth=1.5,
         linestyle=':', label='Random baseline')
ax1.fill_between(depths * 100, gains * 100, depths * 100,
                 alpha=0.12, color=TEAL)
ax1.axvline(10, color=AMBER, linewidth=1.5, linestyle='--', alpha=0.8)
ax1.set_xlabel('Percent of list contacted (%)')
ax1.set_ylabel('Percent of responders captured (%)')
ax1.set_title('Cumulative Gains Chart', fontweight='bold', color=NAVY)
ax1.legend()
ax1.set_xlim(0, 100)
ax1.set_ylim(0, 102)

# Lift chart
ax2.plot(depths[1:] * 100, lift[1:], color=MINT, linewidth=2.5)
ax2.axhline(1.0, color=GRAY, linewidth=1.5, linestyle=':', label='Random baseline (lift = 1)')
ax2.axvline(10, color=AMBER, linewidth=1.5, linestyle='--',
            label='Top decile (10%)')
ax2.set_xlabel('Percent of list contacted (%)')
ax2.set_ylabel('Lift')
ax2.set_title('Lift Chart', fontweight='bold', color=NAVY)
ax2.legend()
ax2.set_xlim(0, 100)
ax2.set_ylim(0, max(lift[1:]) * 1.15)

plt.tight_layout()
plt.show()

### Concept 5.5 -- Threshold Selection

**Business context:** The default classification threshold of 0.50 is rarely optimal.
The correct threshold depends on the *asymmetry between the costs of false positives
and false negatives* in the specific business context.

**Scenario A -- Cancer screening:**
A false negative (missed malignancy) may be fatal. A false positive (unnecessary
biopsy) is costly and stressful but not life-threatening. Decision: lower the
threshold. Recall rises; precision falls. The asymmetric cost of FN justifies the
trade-off.

**Scenario B -- Spam filter:**
A false positive (legitimate email sent to spam) causes the user to miss an important
message. A false negative (spam in the inbox) is recoverable. Decision: raise the
threshold. Precision rises; recall falls. The asymmetric cost of FP justifies
letting some spam reach the inbox.

**General rule:**
- FN cost > FP cost: lower the threshold (prioritise recall).
- FP cost > FN cost: raise the threshold (prioritise precision).

In [ ]:
# -- 5.5 Threshold Selection --------------------------------------------------

# Shared synthetic binary classification dataset
X_th, y_th = make_classification(
    n_samples=3000, n_features=10, n_informative=6,
    n_redundant=2, weights=[0.85, 0.15], random_state=7
)
X_tr, X_ts, y_tr, y_ts = train_test_split(X_th, y_th, test_size=0.3,
                                           random_state=7, stratify=y_th)

clf = GradientBoostingClassifier(n_estimators=80, random_state=7)
clf.fit(X_tr, y_tr)
prob = clf.predict_proba(X_ts)[:, 1]

# Precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y_ts, prob)

# -- Scenario A: cancer screening -- minimise FN (maximise recall) ------------
target_recall = 0.90
valid_a = thresholds[recalls[:-1] >= target_recall]
thresh_a = valid_a[-1] if len(valid_a) > 0 else thresholds[np.argmax(recalls[:-1])]
idx_a    = np.searchsorted(thresholds, thresh_a)
prec_a, rec_a = precisions[idx_a], recalls[idx_a]

# -- Scenario B: spam filter -- minimise FP (maximise precision) --------------
target_prec = 0.92
valid_b  = thresholds[precisions[:-1] >= target_prec]
thresh_b = valid_b[0] if len(valid_b) > 0 else thresholds[np.argmax(precisions[:-1])]
idx_b    = np.searchsorted(thresholds, thresh_b)
prec_b, rec_b = precisions[idx_b], recalls[idx_b]

print('Threshold Analysis -- Two Business Scenarios')
print('=' * 58)
print()
print('Scenario A: Cancer Screening (minimise false negatives)')
print(f'  Optimal threshold: {thresh_a:.3f}  (lowered from 0.50)')
print(f'  Precision: {prec_a:.3f}  ({prec_a*100:.1f}% of flagged cases are true positives)')
print(f'  Recall:    {rec_a:.3f}  ({rec_a*100:.1f}% of actual positives are caught)')
print()
print('Scenario B: Spam Filter (minimise false positives)')
print(f'  Optimal threshold: {thresh_b:.3f}  (raised from 0.50)')
print(f'  Precision: {prec_b:.3f}  ({prec_b*100:.1f}% of flagged emails are genuine spam)')
print(f'  Recall:    {rec_b:.3f}  ({rec_b*100:.1f}% of spam is caught)')
print()
print('General rule:')
print('  FN cost > FP cost --> lower threshold --> recall rises, precision falls.')
print('  FP cost > FN cost --> raise threshold --> precision rises, recall falls.')

# -- Precision-recall curve with both operating points marked -----------------
fig, ax = plt.subplots(figsize=(8, 5.5))

ax.plot(recalls[:-1], precisions[:-1], color=TEAL, linewidth=2.5, label='Model')
ax.scatter([rec_a], [prec_a], color=RED,  s=150, zorder=5,
           label=f'Scenario A -- Cancer (thresh={thresh_a:.2f})\n'
                 f'Recall={rec_a:.2f}, Precision={prec_a:.2f}')
ax.scatter([rec_b], [prec_b], color=AMBER, s=150, zorder=5,
           label=f'Scenario B -- Spam (thresh={thresh_b:.2f})\n'
                 f'Recall={rec_b:.2f}, Precision={prec_b:.2f}')

ax.set_xlabel('Recall  (TPR -- fraction of positives caught)')
ax.set_ylabel('Precision  (fraction of positive predictions that are correct)')
ax.set_title('Precision-Recall Curve: Optimal Threshold by Business Context',
             fontweight='bold', color=NAVY)
ax.legend(loc='lower left', fontsize=8.5)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)

ax.annotate('Lower threshold\n(recall priority)', xy=(rec_a + 0.01, prec_a - 0.06),
            fontsize=8, color=RED)
ax.annotate('Higher threshold\n(precision priority)', xy=(rec_b - 0.22, prec_b + 0.03),
            fontsize=8, color=AMBER)

plt.tight_layout()
plt.show()

---
## Module 6 -- Ch. 3 & 8: Data Wrangling and Visualization

Concepts 6.1 through 6.5 cover the six data quality dimensions, the four-stage
data wrangling pipeline, fitness for purpose, chart selection, and the Alteryx
Designer Cloud pipeline stages. Questions in this module test analytical judgment
rather than calculation.

### Concept 6.1 -- Six Data Quality Dimensions

**Business context:** Poor data quality propagates errors into every downstream
analytical product. Understanding which dimension is violated -- and what the
business consequence is -- is prerequisite to any remediation plan.

**Scenario:** A regional bank's customer database contains six distinct errors,
one per dimension:

| Dimension | Error in this scenario | Business consequence |
|-----------|----------------------|----------------------|
| **Accuracy** | Interest rate charged does not match the loan agreement | Regulatory exposure, customer disputes |
| **Completeness** | Transaction amount blank for 12% of records | Cannot compute total exposure or revenue |
| **Consistency** | Customer name spelled differently across loan and deposit systems | Identity matching fails; duplicate profiles created |
| **Timeliness** | Address not updated since 2019 despite a known move | Mailed statements returned; compliance failures |
| **Uniqueness** | Two records exist for the same customer | Double-counting in reports; duplicate communications |
| **Validity** | Date of birth recorded as 1850-03-14 | Downstream age calculations produce nonsense |

**Exam rule:** Name the violated dimension and state the specific business consequence.
A single record can violate multiple dimensions simultaneously.

In [ ]:
# -- 6.1 Six Data Quality Dimensions ------------------------------------------

# Construct the bank customer database with one error per dimension
records = {
    'customer_id':    ['C001', 'C002', 'C003', 'C004', 'C005', 'C005'],
    'name':           ['Alice Wong', 'Bob Garcia', 'Carol Smith',
                       'David Lee',  'Eve Park',   'EVE PARK'],
    'dob':            ['1985-06-12', '1990-03-28', '1850-03-14',
                       '1978-11-04', '2001-09-17', '2001-09-17'],
    'address':        ['14 Main St', '29 Oak Ave', '7 Elm Rd',
                       '52 Pine Blvd', '88 Cedar Ln', '88 Cedar Ln'],
    'address_since':  ['2023', '2022', '2021', '2019', '2024', '2024'],
    'txn_amount':     [1200.00, None, 350.00, 890.00, 75.00, 75.00],
    'rate_agreed':    [0.045, 0.052, 0.031, 0.039, 0.058, 0.058],
    'rate_charged':   [0.045, 0.052, 0.031, 0.047, 0.058, 0.058],
}

df = pd.DataFrame(records)
print('Raw customer database:')
print(df.to_string(index=False))
print()

# -- Programmatic detection of each dimension violation -----------------------

violations = []

# Accuracy: rate_charged does not match rate_agreed
acc_mask = df['rate_agreed'] != df['rate_charged']
for _, row in df[acc_mask].iterrows():
    violations.append({
        'Customer': row['customer_id'],
        'Dimension': 'Accuracy',
        'Description': f"Rate agreed={row['rate_agreed']:.3f} but charged={row['rate_charged']:.3f}"
    })

# Completeness: missing transaction amount
comp_mask = df['txn_amount'].isna()
for _, row in df[comp_mask].iterrows():
    violations.append({
        'Customer': row['customer_id'],
        'Dimension': 'Completeness',
        'Description': 'Transaction amount is null'
    })

# Consistency: duplicate customer_id with different name formatting
dup_ids = df[df.duplicated('customer_id', keep=False)]
for cid, grp in dup_ids.groupby('customer_id'):
    names = grp['name'].unique()
    if len(names) > 1:
        violations.append({
            'Customer': cid,
            'Dimension': 'Consistency',
            'Description': f"Name variations: {list(names)}"
        })

# Timeliness: address not updated since 2019
tim_mask = df['address_since'].astype(int) <= 2019
for _, row in df[tim_mask & ~df.duplicated('customer_id')].iterrows():
    violations.append({
        'Customer': row['customer_id'],
        'Dimension': 'Timeliness',
        'Description': f"Address unchanged since {row['address_since']}"
    })

# Uniqueness: duplicate customer_id records
uniq_mask = df.duplicated('customer_id')
for _, row in df[uniq_mask].iterrows():
    violations.append({
        'Customer': row['customer_id'],
        'Dimension': 'Uniqueness',
        'Description': 'Duplicate customer record detected'
    })

# Validity: date of birth before 1900
for _, row in df[~df.duplicated('customer_id')].iterrows():
    try:
        dob_year = int(row['dob'][:4])
        if dob_year < 1900:
            violations.append({
                'Customer': row['customer_id'],
                'Dimension': 'Validity',
                'Description': f"Date of birth '{row['dob']}' is outside valid range"
            })
    except Exception:
        pass

violations_df = pd.DataFrame(violations)
print('Detected violations:')
print(violations_df.to_string(index=False))

# -- Summary bar chart --------------------------------------------------------
dim_counts = violations_df['Dimension'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(dim_counts.index, dim_counts.values,
               color=TEAL, edgecolor='white', linewidth=1.1)
ax.set_xlabel('Number of violations detected')
ax.set_title('Data Quality Violations by Dimension', fontweight='bold', color=NAVY)
for bar in bars:
    ax.text(bar.get_width() + 0.04, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

### Concept 6.2 -- Data Wrangling Pipeline Stages

**Business context:** Raw data from operational sources is rarely ready for analysis.
A structured pipeline ensures that errors are identified before changes are made,
that changes are intentional and documented, and that the final output is validated
before it drives any business decision.

**Scenario:** A logistics company merges daily shipment files from three regional
carriers. Carrier B uses a different date format. Eight percent of shipment weights
are recorded as zero. One shipment ID appears in two carrier files.

**Four stages in order:**
1. **Profiling:** Examine structure and anomalies *before* making any changes.
2. **Cleansing:** Correct errors, remove duplicates, handle missing values.
3. **Transformation:** Reshape, aggregate, and derive new fields.
4. **Validation:** Confirm outputs conform to expected rules before downstream use.

**Consequence of skipping Validation:** A time-zone conversion error that produces
negative transit times propagates undetected into performance metrics.

**Exam rule:** Sequence the four stages. Explain what type of error each stage
catches if it is skipped.

In [ ]:
# -- 6.2 Data Wrangling Pipeline -----------------------------------------------

# -- Stage 0: Raw data from three carriers (simulate the raw files) -----------

carrier_a = pd.DataFrame({
    'shipment_id':   ['S001', 'S002', 'S003', 'S004'],
    'ship_date':     ['2025-03-01', '2025-03-01', '2025-03-02', '2025-03-02'],
    'delivery_date': ['2025-03-04', '2025-03-05', '2025-03-06', '2025-03-05'],
    'weight_kg':     [12.5, 0.0, 8.3, 22.1],    # S002 has zero weight
    'carrier':       ['CarrierA'] * 4,
})

carrier_b = pd.DataFrame({
    'shipment_id':   ['S005', 'S006', 'S004'],   # S004 duplicated
    'ship_date':     ['03/01/2025', '03/02/2025', '03/02/2025'],  # different format
    'delivery_date': ['03/03/2025', '03/06/2025', '03/05/2025'],
    'weight_kg':     [5.0, 0.0, 22.1],
    'carrier':       ['CarrierB'] * 3,
})

carrier_c = pd.DataFrame({
    'shipment_id':   ['S007', 'S008'],
    'ship_date':     ['2025-03-01', '2025-03-02'],
    'delivery_date': ['2025-03-03', '2025-03-04'],
    'weight_kg':     [9.8, 14.2],
    'carrier':       ['CarrierC'] * 2,
})

raw_all = pd.concat([carrier_a, carrier_b, carrier_c], ignore_index=True)

print('=' * 60)
print('STAGE 1 -- PROFILING: examine before changing anything')
print('=' * 60)
print(f'Total records across all carriers: {len(raw_all)}')
print(f'Unique shipment IDs:               {raw_all["shipment_id"].nunique()}')
print(f'Duplicate shipment IDs:            {raw_all.duplicated("shipment_id").sum()}')
print(f'Zero-weight records:               {(raw_all["weight_kg"] == 0).sum()}')
print(f'Date formats detected:             mixed (YYYY-MM-DD and MM/DD/YYYY)')
print()

print('=' * 60)
print('STAGE 2 -- CLEANSING: standardise, deduplicate, flag anomalies')
print('=' * 60)

def parse_date_flexible(s):
    # Parses both YYYY-MM-DD and MM/DD/YYYY date formats.
    try:
        return pd.to_datetime(s, format='%Y-%m-%d')
    except ValueError:
        return pd.to_datetime(s, format='%m/%d/%Y')

raw_all['ship_date_clean']     = raw_all['ship_date'].apply(parse_date_flexible)
raw_all['delivery_date_clean'] = raw_all['delivery_date'].apply(parse_date_flexible)

# Flag zero-weight records (do not impute silently -- flag for review)
raw_all['weight_flagged'] = raw_all['weight_kg'] == 0

# Deduplicate: keep CarrierA record where S004 appears in both A and B
clean = raw_all.drop_duplicates(subset='shipment_id', keep='first').copy()

print(f'Records after deduplication: {len(clean)}  (removed {len(raw_all)-len(clean)} duplicate)')
print(f'Zero-weight records flagged: {clean["weight_flagged"].sum()}')
print()

print('=' * 60)
print('STAGE 3 -- TRANSFORMATION: derive transit time and aggregate')
print('=' * 60)

clean['transit_days'] = (clean['delivery_date_clean'] -
                         clean['ship_date_clean']).dt.days

daily_summary = (clean[~clean['weight_flagged']]
                 .groupby('ship_date_clean')
                 .agg(shipments=('shipment_id', 'count'),
                      avg_transit=('transit_days', 'mean'),
                      total_weight=('weight_kg', 'sum'))
                 .reset_index()
                 .rename(columns={'ship_date_clean': 'date'}))

print('Daily summary (zero-weight records excluded):')
print(daily_summary.to_string(index=False))
print()

print('=' * 60)
print('STAGE 4 -- VALIDATION: confirm output conforms to rules')
print('=' * 60)

# Rule 1: no negative transit times
neg_transit = clean[clean['transit_days'] < 0]
print(f'Rule 1 -- No negative transit times:   '
      f'{"PASS" if len(neg_transit)==0 else "FAIL -- " + str(len(neg_transit)) + " violations"}')

# Rule 2: record count matches source totals (after deduplication)
expected_unique = raw_all['shipment_id'].nunique()
print(f'Rule 2 -- Unique shipment count:       '
      f'{"PASS" if len(clean)==expected_unique else "FAIL"}  '
      f'({len(clean)} records, expected {expected_unique})')

# Rule 3: no carrier codes outside approved list
approved_carriers = {'CarrierA', 'CarrierB', 'CarrierC'}
unknown = set(clean['carrier'].unique()) - approved_carriers
print(f'Rule 3 -- No unknown carrier codes:    '
      f'{"PASS" if len(unknown)==0 else "FAIL -- " + str(unknown)}')

print()
print('All validation rules passed -- pipeline output is ready for analysis.')

### Concept 6.3 -- Fitness for Purpose

**Business context:** A dataset that is suitable for one analytical task may be
entirely unsuitable for another. Fitness for purpose is assessed *relative to the
specific question being asked*, not as a global property of the dataset.

**Scenario:** A hospital collects patient satisfaction survey responses from
2018 through 2022. The dataset contains one row per discharged patient, with
columns for department ratings, wait time ratings, communication ratings, and
overall satisfaction score.

**Study A -- Trend analysis of patient satisfaction over time:** The dataset is
*fit for purpose*. It covers the relevant institution, the time period of interest,
and the questions are consistent across years, enabling trend analysis.

**Study B -- Predict 30-day readmission:** The dataset is *not fit for purpose*.
Predicting readmission requires clinical variables: diagnosis codes, length of stay,
medication reconciliation status, comorbidity scores. Satisfaction scores do not
contain this information. A model trained on satisfaction scores alone will have
negligible predictive power for readmission, regardless of how well the data is cleaned.

**Exam rule:** The same dataset can be fit for one task and unfit for another.
State the specific missing features or structural gap, not merely "the data is
insufficient."

In [ ]:
# -- 6.3 Fitness for Purpose ---------------------------------------------------

import warnings
warnings.filterwarnings('ignore')

np.random.seed(99)
n_patients = 800

# Simulate patient satisfaction dataset (2018-2022)
years = np.random.choice(range(2018, 2023), n_patients)
satisfaction_df = pd.DataFrame({
    'patient_id':     range(n_patients),
    'year':           years,
    'dept_rating':    np.clip(np.random.normal(3.8, 0.7, n_patients), 1, 5).round(1),
    'wait_rating':    np.clip(np.random.normal(3.2, 0.9, n_patients), 1, 5).round(1),
    'comm_rating':    np.clip(np.random.normal(4.0, 0.6, n_patients), 1, 5).round(1),
    'overall_score':  np.clip(np.random.normal(3.8, 0.7, n_patients), 1, 5).round(1),
    # True readmission label (NOT correlated with satisfaction scores)
    'readmitted_30d': np.random.choice([0, 1], n_patients, p=[0.85, 0.15]),
})

print('Dataset summary:')
print(f'  Rows: {len(satisfaction_df)}, Years: {sorted(satisfaction_df["year"].unique())}')
print(f'  Columns: {list(satisfaction_df.columns)}')
print(f'  Readmission rate: {satisfaction_df["readmitted_30d"].mean():.1%}')
print()

# -- Study A: Trend analysis -- FIT FOR PURPOSE --------------------------------
print('=' * 58)
print('STUDY A: Patient Satisfaction Trend Analysis')
print('Assessment: FIT FOR PURPOSE')
print('=' * 58)

yearly_avg = (satisfaction_df.groupby('year')['overall_score']
              .mean().reset_index()
              .rename(columns={'overall_score': 'avg_satisfaction'}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(yearly_avg['year'], yearly_avg['avg_satisfaction'],
             color=TEAL, linewidth=2.5, marker='o', markersize=8)
axes[0].set_ylim(1, 5)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Average overall satisfaction score')
axes[0].set_title('Study A: Satisfaction Trend 2018-2022\n(Fit for purpose)',
                  fontweight='bold', color=NAVY)
axes[0].set_xticks(yearly_avg['year'])

print(yearly_avg.to_string(index=False))
print()

# -- Study B: Readmission prediction -- NOT FIT FOR PURPOSE -------------------
print('=' * 58)
print('STUDY B: 30-Day Readmission Prediction')
print('Assessment: NOT FIT FOR PURPOSE')
print('=' * 58)

features = ['dept_rating', 'wait_rating', 'comm_rating', 'overall_score']
X_sat = satisfaction_df[features].values
y_sat = satisfaction_df['readmitted_30d'].values

X_tr2, X_ts2, y_tr2, y_ts2 = train_test_split(X_sat, y_sat, test_size=0.3,
                                                random_state=42, stratify=y_sat)
clf2 = RandomForestClassifier(n_estimators=100, random_state=42)
clf2.fit(X_tr2, y_tr2)
prob2 = clf2.predict_proba(X_ts2)[:, 1]
fpr2, tpr2, _ = roc_curve(y_ts2, prob2)
auc2 = auc(fpr2, tpr2)

axes[1].plot(fpr2, tpr2, color=RED, linewidth=2.5,
             label=f'Satisfaction-based model (AUC = {auc2:.3f})')
axes[1].plot([0, 1], [0, 1], color=GRAY, linewidth=1.5, linestyle=':',
             label='Random model (AUC = 0.50)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Study B: Readmission Prediction ROC\n(Not fit for purpose)',
                  fontweight='bold', color=RED)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'AUC of satisfaction-based readmission model: {auc2:.3f}')
print(f'  AUC near 0.50 confirms the model has no discriminative power.')
print(f'  The data is analytically sound but lacks the required clinical features:')
print(f'  diagnosis codes, length of stay, comorbidity indices, medication status.')
print()
print('Conclusion:')
print('  The SAME dataset is fit for Study A and unfit for Study B.')
print('  Fitness for purpose is task-relative, not a global property of the data.')

### Concept 6.4 -- Chart Selection

**Business context:** Chart type must match the data relationship being communicated.
A mismatched chart obscures the message and may lead decision-makers to incorrect
conclusions. The most common error in business reporting is using a pie chart to
compare values across many categories.

**Scenario:** A marketing director receives a pie chart with 12 slices representing
market share across 12 product categories.

**What is wrong:** Human perception cannot accurately compare 12 angles or areas,
especially when several slices are close in size. The pie chart principle is that
it should be used *only when the primary message is part-to-whole composition* and
the number of categories is four to five at most.

**Correct alternative:** A horizontal bar chart sorted by value. Bar length is
accurately perceived and magnitude differences that are invisible in a pie chart
become immediately apparent.

**Chart selection reference:**

| Data relationship | Appropriate chart type |
|-------------------|----------------------|
| Trend of a continuous variable over time | Line chart |
| Comparison across discrete categories | Bar chart |
| Relationship between two continuous variables | Scatter plot |
| Part-to-whole, four or fewer categories | Pie or donut chart |
| Distribution of a single continuous variable | Histogram or box plot |
| Ranking with precise values | Horizontal bar chart, sorted |

**Exam rule:** Identify the violated principle by name and specify the correct chart.

In [ ]:
# -- 6.4 Chart Selection -- Pie vs. Bar ----------------------------------------

# Twelve product categories with simulated market share
categories = [
    'Office Supplies', 'Electronics', 'Furniture', 'Apparel',
    'Home Goods',      'Sports',      'Toys',       'Automotive',
    'Garden',          'Books',        'Health',    'Food & Beverage'
]
shares = np.array([18, 15, 13, 11, 9, 8, 7, 6, 5, 4, 3, 1], dtype=float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# -- Panel 1: Pie chart (THE WRONG CHOICE FOR 12 CATEGORIES) ------------------
wedge_colors = plt.cm.tab20(np.linspace(0, 1, len(categories)))
ax1.pie(shares, labels=categories, colors=wedge_colors,
        autopct='%1.0f%%', pctdistance=0.78, startangle=90,
        textprops={'fontsize': 7})
ax1.set_title('WRONG: Pie Chart (12 categories)\n'
              'Angles are difficult to compare accurately',
              fontweight='bold', color=RED, pad=14)

ax1.text(0, -1.45, 'Which is larger: Garden (5%) or Books (4%)?\nThe angles are nearly indistinguishable.',
         ha='center', fontsize=9, color=RED, fontstyle='italic')

# -- Panel 2: Horizontal bar chart (THE CORRECT CHOICE) -----------------------
sort_idx = np.argsort(shares)
sorted_cats   = [categories[i] for i in sort_idx]
sorted_shares = shares[sort_idx]
bar_colors    = [MINT if s > 10 else TEAL for s in sorted_shares]

bars = ax2.barh(sorted_cats, sorted_shares, color=bar_colors,
                edgecolor='white', linewidth=0.8)
ax2.set_xlabel('Market share (%)')
ax2.set_title('CORRECT: Horizontal Bar Chart (sorted)\n'
              'Bar length is accurately perceived',
              fontweight='bold', color=NAVY, pad=14)
ax2.set_xlim(0, 22)

for bar, val in zip(bars, sorted_shares):
    ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.0f}%', va='center', fontsize=8.5, fontweight='bold', color=NAVY)

ax2.text(19, 1, 'Garden vs. Books:\n5% vs. 4%\nclearly visible',
         fontsize=8, color=TEAL, fontstyle='italic',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8F6F3', edgecolor=TEAL))

plt.suptitle('Chart Selection: 12-Category Market Share Comparison',
             fontsize=13, fontweight='bold', color=NAVY, y=1.01)
plt.tight_layout()
plt.show()

# -- Reference table ----------------------------------------------------------
print()
print('Chart selection reference:')
ref_data = {
    'Data relationship': [
        'Trend of a continuous variable over time',
        'Comparison across discrete categories',
        'Relationship between two continuous variables',
        'Part-to-whole (four or fewer categories)',
        'Distribution of a single continuous variable',
        'Ranking with precise values',
    ],
    'Correct chart type': [
        'Line chart',
        'Bar chart (vertical or horizontal)',
        'Scatter plot',
        'Pie or donut chart',
        'Histogram or box plot',
        'Horizontal bar chart, sorted',
    ]
}
ref_df = pd.DataFrame(ref_data)
print(ref_df.to_string(index=False))

### Concept 6.5 -- Alteryx Designer Cloud Pipeline Stages

**Business context:** Alteryx Designer Cloud organises data preparation work into
four pipeline stages that correspond to the Maveryx Academy curriculum. Each stage
serves a distinct purpose and maps to specific tool categories within the platform.

**Scenario:** A sales operations analyst prepares a weekly revenue report from a
raw CSV file that requires joining to a product master table, removing test records,
computing revenue, and aggregating by region and category.

**Four stages:**

| Stage | Purpose | Example Alteryx tools |
|-------|---------|----------------------|
| Data Preparation | Connect to sources; inspect structure and quality | Input Data, Browse, Select |
| Combining and Cleansing | Join sources; standardise; remove duplicates; handle nulls | Join, Data Cleanse, Filter, Unique |
| Advanced Data Preparation | Compute derived fields; aggregate; parse; reformat | Formula, Summarize, Text To Columns, DateTime |
| Workflow Design | Organise for reliability, reuse, and documentation; confirm output schema | Comment, Tool Container, Output Data |

**Critical scope boundary:** R-based Predictive Tools (regression, decision trees,
scoring) are available only in Alteryx Designer *Desktop*, not in Designer Cloud.
Any exam question about predictive modelling in Alteryx assumes Desktop unless
the question explicitly specifies Cloud.

**Exam rule:** Identify which stage a described operation belongs to. Recognise
that Designer Cloud and Designer Desktop are not interchangeable for predictive work.

In [ ]:
# -- 6.5 Alteryx Designer Cloud Pipeline -- Python Equivalent -----------------
# This code mirrors the four Maveryx Academy pipeline stages using pandas.
# The comments identify which Alteryx tool category each operation maps to.

# -- STAGE 1: DATA PREPARATION ------------------------------------------------
# Alteryx equivalents: Input Data tool, Browse tool, Select tool

print('=' * 62)
print('STAGE 1 -- DATA PREPARATION: connect, inspect, identify issues')
print('=' * 62)

np.random.seed(21)
n = 120

raw_sales = pd.DataFrame({
    'txn_id':      range(1, n + 1),
    'store_id':    np.random.choice([101, 102, 103, 999], n,
                                    p=[0.40, 0.35, 0.20, 0.05]),  # 999 = test store
    'product_code': np.random.choice(['  P01', 'P02', 'P03', 'P04'], n),  # leading spaces
    'units_sold':   np.random.randint(1, 50, n),
    'unit_price':   np.random.choice([9.99, 24.99, 49.99, 99.99], n),
    'discount_pct': np.random.choice([0.0, 0.05, 0.10, None], n,
                                      p=[0.60, 0.20, 0.15, 0.05]),
    'region':       np.random.choice(['Northeast', 'Southeast', 'West', None], n,
                                      p=[0.35, 0.30, 0.30, 0.05]),
})

product_master = pd.DataFrame({
    'product_code': ['P01', 'P02', 'P03', 'P04'],
    'category':     ['Office Supplies', 'Electronics', 'Furniture', 'Apparel'],
})

print(f'Raw file rows:       {len(raw_sales)}')
print(f'Null discount_pct:   {raw_sales["discount_pct"].isna().sum()}')
print(f'Null region:         {raw_sales["region"].isna().sum()}')
print(f'Test store (999):    {(raw_sales["store_id"] == 999).sum()} records')
print(f'Leading spaces in product_code: '
      f'{raw_sales["product_code"].str.startswith(" ").sum()} records')
print()

# -- STAGE 2: COMBINING AND CLEANSING -----------------------------------------
# Alteryx equivalents: Join tool, Data Cleanse tool, Filter tool, Unique tool

print('=' * 62)
print('STAGE 2 -- COMBINING AND CLEANSING')
print('=' * 62)

cleansed = raw_sales.copy()

# Data Cleanse: strip leading/trailing spaces from product codes
cleansed['product_code'] = cleansed['product_code'].str.strip()

# Data Cleanse: fill null discount_pct with 0 (no discount applied)
cleansed['discount_pct'] = cleansed['discount_pct'].fillna(0.0)

# Data Cleanse: fill null region with 'Unknown'
cleansed['region'] = cleansed['region'].fillna('Unknown')

# Filter: remove test store transactions
n_before = len(cleansed)
cleansed = cleansed[cleansed['store_id'] != 999].copy()
print(f'Test records removed: {n_before - len(cleansed)}  ({n_before} -> {len(cleansed)} rows)')

# Join: merge with product master on product_code
cleansed = cleansed.merge(product_master, on='product_code', how='left')
print(f'Joined to product master -- categories added: {cleansed["category"].notna().sum()}')
print()

# -- STAGE 3: ADVANCED DATA PREPARATION ---------------------------------------
# Alteryx equivalents: Formula tool, Summarize tool

print('=' * 62)
print('STAGE 3 -- ADVANCED DATA PREPARATION: derive fields and aggregate')
print('=' * 62)

# Formula tool equivalent: compute revenue
cleansed['revenue'] = (cleansed['units_sold'] *
                       cleansed['unit_price'] *
                       (1 - cleansed['discount_pct']))

# Summarize tool equivalent: aggregate to weekly totals by region and category
weekly_summary = (cleansed
                  .groupby(['region', 'category'], dropna=False)
                  .agg(transactions=('txn_id', 'count'),
                       total_units=('units_sold', 'sum'),
                       total_revenue=('revenue', 'sum'))
                  .reset_index()
                  .sort_values('total_revenue', ascending=False))

print('Weekly revenue summary by region and category:')
print(weekly_summary.to_string(index=False, float_format='${:,.2f}'.format))
print()

# -- STAGE 4: WORKFLOW DESIGN / VALIDATION ------------------------------------
# Alteryx equivalents: Output Data tool, Comment tools, Tool Container

print('=' * 62)
print('STAGE 4 -- WORKFLOW DESIGN AND VALIDATION: confirm output schema')
print('=' * 62)

def validate_output(df, source_df):
    results = []

    # Rule 1: no negative revenue values
    neg_rev = (df['total_revenue'] < 0).sum()
    results.append(('No negative revenue',
                    'PASS' if neg_rev == 0 else f'FAIL ({neg_rev} violations)'))

    # Rule 2: transaction count should match cleansed record count
    total_txns = df['transactions'].sum()
    results.append(('Transaction count matches cleansed file',
                    f'PASS ({total_txns} = {len(source_df)})' if total_txns == len(source_df)
                    else f'FAIL ({total_txns} vs {len(source_df)})'))

    # Rule 3: no null category values (every product code had a match in master)
    null_cats = df['category'].isna().sum()
    results.append(('All categories resolved',
                    'PASS' if null_cats == 0 else f'FAIL ({null_cats} nulls)'))

    for rule, status in results:
        print(f'  {rule:<42}  {status}')

validate_output(weekly_summary, cleansed)
print()
print('Scope boundary reminder:')
print('  The operations above are all available in Alteryx Designer CLOUD.')
print('  R-based Predictive Tools (regression, scoring, decision trees)')
print('  are available in Designer DESKTOP only.')

# -- Stage summary bar chart --------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 3.5))
stage_labels = ['Stage 1\nData\nPreparation',
                'Stage 2\nCombining &\nCleansing',
                'Stage 3\nAdvanced\nPreparation',
                'Stage 4\nWorkflow\nDesign']
stage_ops    = [3, 5, 2, 3]   # approximate number of operations per stage
stage_cols   = [NAVY, TEAL, MINT, AMBER]

bars = ax.bar(stage_labels, stage_ops, color=stage_cols,
              edgecolor='white', linewidth=1.2, width=0.55)
ax.set_ylabel('Number of operations (illustrative)')
ax.set_title('Alteryx Designer Cloud: Four Pipeline Stages',
             fontweight='bold', color=NAVY)
ax.set_ylim(0, 7)
for bar, val in zip(bars, stage_ops):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(val), ha='center', fontsize=11, fontweight='bold', color=NAVY)
plt.tight_layout()
plt.show()



---



<font color = blue><i>note: This notebook is for illustrative purposes only and may include unintentional errors or ommissions. This notebook is intended to be used as a refresher resource and not as a primary source or reference.  In other words, "Good luck!".</i></font>



---

